# RAG Evaluation — LLM as Judge

Evaluate the quality of end-to-end RAG answers using two approaches:
1. **Cosine similarity** — compare generated answer to expected answer
2. **LLM-as-judge** — use gpt-4o-mini to rate relevance

We also compare two prompt strategies:
- **Prompt v1** — basic prompt without source attribution
- **Prompt v2** — prompt with explicit source + tool context

Prerequisites: `evaluation-data-generation.ipynb` must have been run.

In [ ]:
import os
import sys
import json

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from openai import OpenAI
from sentence_transformers import SentenceTransformer

sys.path.insert(0, "../dataops_assistant")
from rag import vector_search, hybrid_search

client = OpenAI()
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Setup complete.")

In [ ]:
df_ground_truth = pd.read_csv("../data/ground-truth-retrieval.csv")
# Sample for cost control
SAMPLE_SIZE = 100
df_sample = df_ground_truth.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Evaluating on {len(df_sample)} questions")
df_sample.head()

## Prompt variants

In [ ]:
PROMPT_V1 = """
Answer the QUESTION based on the CONTEXT.
Use only the facts from the CONTEXT.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

PROMPT_V2 = """
You are a DataOps expert assistant specializing in dbt, Apache Airflow, and Great Expectations.
Answer the QUESTION using only information from the CONTEXT below.
If the answer is not in the context, say "I don't have enough information to answer this question."
Always mention which tool (dbt/Airflow/Great Expectations) your answer refers to.

QUESTION: {question}

CONTEXT:
{context}
""".strip()


def build_context(search_results: list) -> str:
    return "\n\n".join(
        f"[Source: {r.get('source', 'unknown')}]\n{r['text']}"
        for r in search_results
    )


def ask_llm(prompt: str, model: str = "gpt-4o-mini") -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

## Generate answers for both prompt variants

In [ ]:
def generate_answers(df: pd.DataFrame, prompt_template: str, label: str) -> pd.DataFrame:
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=label):
        search_results = hybrid_search(row["question"])
        context = build_context(search_results)
        prompt = prompt_template.format(question=row["question"], context=context)
        answer = ask_llm(prompt)
        records.append({
            "id": row["id"],
            "question": row["question"],
            "answer": answer,
            "prompt_version": label,
        })
    return pd.DataFrame(records)


df_v1 = generate_answers(df_sample, PROMPT_V1, "v1_basic")
df_v2 = generate_answers(df_sample, PROMPT_V2, "v2_expert")

## LLM-as-Judge evaluation

In [ ]:
JUDGE_PROMPT = """
You are an expert evaluator for a RAG system about DataOps tools.
Analyze the relevance of the generated answer to the question.
Classify as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Question: {question}
Generated Answer: {answer}

Respond with parsable JSON only:
{{"Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT", "Explanation": "..."}}
""".strip()


def evaluate_with_llm(df: pd.DataFrame) -> pd.DataFrame:
    evaluations = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="LLM judge"):
        prompt = JUDGE_PROMPT.format(question=row["question"], answer=row["answer"])
        raw = ask_llm(prompt)
        try:
            result = json.loads(raw)
        except json.JSONDecodeError:
            result = {"Relevance": "UNKNOWN", "Explanation": "parse error"}
        evaluations.append(result)
    df = df.copy()
    df["relevance"] = [e["Relevance"] for e in evaluations]
    df["explanation"] = [e["Explanation"] for e in evaluations]
    return df


df_v1_eval = evaluate_with_llm(df_v1)
df_v2_eval = evaluate_with_llm(df_v2)

## Cosine similarity evaluation

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def add_cosine_scores(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    q_embeddings = sentence_model.encode(df["question"].tolist())
    a_embeddings = sentence_model.encode(df["answer"].tolist())
    scores = [
        cosine_similarity(q, a)
        for q, a in zip(q_embeddings, a_embeddings)
    ]
    df["cosine_score"] = scores
    return df


df_v1_eval = add_cosine_scores(df_v1_eval)
df_v2_eval = add_cosine_scores(df_v2_eval)

# Save results
df_v1_eval.to_csv("../data/rag-eval-v1.csv", index=False)
df_v2_eval.to_csv("../data/rag-eval-v2.csv", index=False)
print("Results saved.")

## Final comparison

In [ ]:
def summarize_eval(df: pd.DataFrame, label: str) -> dict:
    relevance_counts = df["relevance"].value_counts(normalize=True)
    return {
        "prompt": label,
        "RELEVANT %": round(relevance_counts.get("RELEVANT", 0) * 100, 1),
        "PARTLY_RELEVANT %": round(relevance_counts.get("PARTLY_RELEVANT", 0) * 100, 1),
        "NON_RELEVANT %": round(relevance_counts.get("NON_RELEVANT", 0) * 100, 1),
        "avg_cosine": round(df["cosine_score"].mean(), 4),
    }


summary = pd.DataFrame([
    summarize_eval(df_v1_eval, "v1 basic"),
    summarize_eval(df_v2_eval, "v2 expert"),
])
summary.set_index("prompt")

In [ ]:
# Recommendation
v1_relevant = df_v1_eval["relevance"].eq("RELEVANT").mean()
v2_relevant = df_v2_eval["relevance"].eq("RELEVANT").mean()

winner = "v2 expert" if v2_relevant >= v1_relevant else "v1 basic"
print(f"\n→ Best prompt for production: {winner}")
print(f"  v1 RELEVANT: {v1_relevant:.1%}")
print(f"  v2 RELEVANT: {v2_relevant:.1%}")
print("\nUpdate PROMPT_TEMPLATE in rag.py accordingly.")